In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.utils import save_image, make_grid
from tqdm import tqdm
import os
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = np.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class ConditionalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_classes, dropout=0.1):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        self.class_emb = nn.Embedding(num_classes + 1, out_ch)
        
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        
        self.dropout = nn.Dropout(dropout)
        self.act = nn.SiLU()
        
        self.residual = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, time_emb, class_labels):
        h = self.conv1(x)
        h = self.norm1(h)
        
        time_emb = self.time_mlp(time_emb)
        class_emb = self.class_emb(class_labels)
        cond = (time_emb + class_emb).unsqueeze(-1).unsqueeze(-1)
        h = h + cond
        
        h = self.act(h)
        h = self.dropout(h)
        
        h = self.conv2(h)
        h = self.norm2(h)
        h = self.act(h)
        
        return h + self.residual(x)

class ConditionalUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, num_classes=4, time_emb_dim=128):
        super().__init__()
        
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim),
        )
        
        self.down1 = ConditionalBlock(in_channels, 64, time_emb_dim, num_classes)
        self.down2 = ConditionalBlock(64, 128, time_emb_dim, num_classes)
        self.down3 = ConditionalBlock(128, 256, time_emb_dim, num_classes)
        
        self.mid = ConditionalBlock(256, 256, time_emb_dim, num_classes)
        
        self.up3 = ConditionalBlock(512, 128, time_emb_dim, num_classes)
        self.up2 = ConditionalBlock(256, 64, time_emb_dim, num_classes)
        self.up1 = ConditionalBlock(128, 64, time_emb_dim, num_classes)
        
        self.out = nn.Conv2d(64, out_channels, 1)
        self.pool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
    def forward(self, x, timestep, class_labels):
        time_emb = self.time_mlp(timestep)
        
        skip1 = self.down1(x, time_emb, class_labels)
        x = self.pool(skip1)
        
        skip2 = self.down2(x, time_emb, class_labels)
        x = self.pool(skip2)
        
        skip3 = self.down3(x, time_emb, class_labels)
        x = self.pool(skip3)
        
        x = self.mid(x, time_emb, class_labels)
        
        x = self.upsample(x)
        x = torch.cat([x, skip3], dim=1)
        x = self.up3(x, time_emb, class_labels)
        
        x = self.upsample(x)
        x = torch.cat([x, skip2], dim=1)
        x = self.up2(x, time_emb, class_labels)
        
        x = self.upsample(x)
        x = torch.cat([x, skip1], dim=1)
        x = self.up1(x, time_emb, class_labels)
        
        return self.out(x)

class DiffusionDiscriminator(nn.Module):
    def __init__(self, img_channels=3, num_classes=4, base_channels=64):
        super().__init__()
        
        self.time_emb_dim = 128
        self.time_emb = SinusoidalPositionEmbeddings(self.time_emb_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(self.time_emb_dim, self.time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(self.time_emb_dim * 2, self.time_emb_dim)
        )
        
        self.class_emb = nn.Embedding(num_classes, num_classes)
        
        def disc_block(in_ch, out_ch, bn=True):
            layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1)]
            if bn:
                layers.append(nn.BatchNorm2d(out_ch))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)
        
        self.model = nn.Sequential(
            disc_block(img_channels + num_classes, base_channels, bn=False),
            disc_block(base_channels, base_channels * 2),
            disc_block(base_channels * 2, base_channels * 4),
            disc_block(base_channels * 4, base_channels * 8),
        )
        
        ds_size = 128 // 16
        self.adv_layer = nn.Sequential(
            nn.Linear(base_channels * 8 * ds_size * ds_size + self.time_emb_dim, 1),
            nn.Sigmoid()
        )
    
    def forward(self, img, t, class_labels):
        t_emb = self.time_emb(t)
        t_emb = self.time_mlp(t_emb)
        
        class_emb = self.class_emb(class_labels)
        class_emb = class_emb.view(class_emb.size(0), class_emb.size(1), 1, 1)
        class_emb = class_emb.expand(class_emb.size(0), class_emb.size(1), img.size(2), img.size(3))
        
        d_in = torch.cat([img, class_emb], dim=1)
        out = self.model(d_in)
        out = out.view(out.shape[0], -1)
        out = torch.cat([out, t_emb], dim=1)
        validity = self.adv_layer(out)
        
        return validity

class DiffusionGANSampler:
    def __init__(self, denoiser, discriminator, num_timesteps=1000):
        self.denoiser = denoiser
        self.discriminator = discriminator
        self.num_timesteps = num_timesteps
        
        steps = num_timesteps + 1
        x = torch.linspace(0, num_timesteps, steps)
        alphas_cumprod = torch.cos(((x / num_timesteps) + 0.008) / (1 + 0.008) * torch.pi * 0.5) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        self.betas = torch.clip(betas, 0.0001, 0.9999).to(device)
        
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        self.posterior_variance = self.betas * (1.0 - F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)) / (1.0 - self.alphas_cumprod)
    
    @torch.no_grad()
    def sample(self, batch_size, class_labels, guidance_scale=0.3, use_discriminator=True):
        x = torch.randn(batch_size, 3, 128, 128).to(device)
        
        for t in tqdm(reversed(range(self.num_timesteps)), desc="Sampling", total=self.num_timesteps, leave=False):
            t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
            
            predicted_noise = self.denoiser(x, t_batch, class_labels)
            
            beta = self.betas[t]
            sqrt_one_minus_alpha_cumprod = self.sqrt_one_minus_alphas_cumprod[t]
            sqrt_recip_alpha = self.sqrt_recip_alphas[t]
            
            model_mean = sqrt_recip_alpha * (x - beta * predicted_noise / sqrt_one_minus_alpha_cumprod)
            
            if use_discriminator and t > 0:
                with torch.enable_grad():
                    x_grad = x.detach().requires_grad_(True)
                    disc_score = self.discriminator(x_grad, t_batch, class_labels)
                    grad = torch.autograd.grad(disc_score.sum(), x_grad)[0]
                
                model_mean = model_mean + guidance_scale * grad * self.posterior_variance[t]
            
            if t > 0:
                noise = torch.randn_like(x)
                x = model_mean + torch.sqrt(self.posterior_variance[t]) * noise
            else:
                x = model_mean
        
        return x

def load_pretrained_diffusion_model():
    print("Loading pre trained diffusion model")
    
    checkpoint = torch.load('results/diffusion/best_model.pth', map_location=device)
    
    denoiser = ConditionalUNet(
        in_channels=3,
        out_channels=3,
        num_classes=4,
        time_emb_dim=128
    ).to(device)
    
    denoiser.load_state_dict(checkpoint['model_state_dict'])
    denoiser.eval()
    
    print("Diffusion model loaded")
    return denoiser

def train_discriminator_for_diffusion(denoiser, minority_classes=[1, 4, 8, 9], epochs=50):
    print("\nTraining discriminator for diffusion GAN hybrid")
    
    data_dir = 'data/processed'
    X_train = np.load(f'{data_dir}/X_train.npy')
    y_train = np.load(f'{data_dir}/y_train.npy')
    
    mask = np.isin(y_train, minority_classes)
    X_train = X_train[mask]
    y_train = y_train[mask]
    
    label_mapping = {old: new for new, old in enumerate(minority_classes)}
    y_train = np.array([label_mapping[label] for label in y_train])
    
    print(f"Training samples: {len(X_train)}")
    
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: (x - 0.5) * 2.0)
    ])
    
    train_images = []
    for img in tqdm(X_train[:800], desc="Preprocessing"):
        img_pil = Image.fromarray(img)
        img_tensor = transform(img_pil)
        train_images.append(img_tensor)
    
    train_images = torch.stack(train_images).to(device)
    train_labels = torch.from_numpy(y_train[:800]).to(device)
    
    discriminator = DiffusionDiscriminator(img_channels=3, num_classes=4).to(device)
    optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.999))
    adversarial_loss = nn.BCELoss()
    
    num_timesteps = 1000
    steps = num_timesteps + 1
    x = torch.linspace(0, num_timesteps, steps)
    alphas_cumprod = torch.cos(((x / num_timesteps) + 0.008) / (1 + 0.008) * torch.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    betas = torch.clip(betas, 0.0001, 0.9999).to(device)
    alphas_cumprod = torch.cumprod(1.0 - betas, dim=0)
    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
    sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
    
    for epoch in range(epochs):
        discriminator.train()
        d_losses = []
        
        num_batches = len(train_images) // 16
        
        for i in range(num_batches):
            batch_images = train_images[i*16:(i+1)*16]
            batch_labels = train_labels[i*16:(i+1)*16]
            batch_size = batch_images.size(0)
            
            t = torch.randint(0, num_timesteps, (batch_size,), device=device).long()
            noise = torch.randn_like(batch_images)
            
            sqrt_alpha_t = sqrt_alphas_cumprod[t][:, None, None, None]
            sqrt_one_minus_alpha_t = sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
            noisy_images = sqrt_alpha_t * batch_images + sqrt_one_minus_alpha_t * noise
            
            with torch.no_grad():
                predicted_noise = denoiser(noisy_images, t, batch_labels)
                fake_images = noisy_images - sqrt_one_minus_alpha_t * predicted_noise
            
            optimizer_D.zero_grad()
            
            real_validity = discriminator(batch_images, t, batch_labels)
            fake_validity = discriminator(fake_images.detach(), t, batch_labels)
            
            real_loss = adversarial_loss(real_validity, torch.ones_like(real_validity) * 0.9)
            fake_loss = adversarial_loss(fake_validity, torch.zeros_like(fake_validity) + 0.1)
            
            d_loss = (real_loss + fake_loss) / 2
            d_loss.backward()
            optimizer_D.step()
            
            d_losses.append(d_loss.item())
        
        avg_d_loss = np.mean(d_losses)
        print(f"Epoch {epoch+1}/{epochs}, D Loss: {avg_d_loss:.4f}")
    
    return discriminator

def save_hybrid_samples(sampler, minority_classes=[1, 4, 8, 9], samples_per_class=8):
    print("\nSaving visual samples")
    
    os.makedirs('results/diffusion_gan/samples', exist_ok=True)
    
    for class_idx, original_class in enumerate(minority_classes):
        print(f"Class {original_class}")
        
        class_labels = torch.full((samples_per_class,), class_idx, device=device, dtype=torch.long)
        
        samples = sampler.sample(samples_per_class, class_labels, 
                                guidance_scale=0.3, use_discriminator=True)
        
        samples = (samples + 1) / 2
        samples = torch.clamp(samples, 0, 1)
        
        grid = make_grid(samples, nrow=4, padding=2)
        save_image(grid, f'results/diffusion_gan/samples/class_{original_class}_samples.png')
        
        for i in range(min(4, samples_per_class)):
            save_image(samples[i], f'results/diffusion_gan/samples/class_{original_class}_img_{i}.png')
    
    print("Samples saved to results/diffusion_gan/samples/")

def generate_hybrid_samples(denoiser, discriminator, minority_classes=[1, 4, 8, 9], 
                           samples_per_class=200, guidance_scale=0.3):
    print(f"\nGenerating hybrid samples with guidance_scale={guidance_scale}")
    
    sampler = DiffusionGANSampler(denoiser, discriminator)
    
    save_hybrid_samples(sampler, minority_classes, samples_per_class=8)
    
    all_images = []
    all_labels = []
    
    for class_idx, original_class in enumerate(minority_classes):
        print(f"Generating class {original_class}")
        
        batch_size = 16
        num_batches = (samples_per_class + batch_size - 1) // batch_size
        
        for batch in range(num_batches):
            current_batch_size = min(batch_size, samples_per_class - batch * batch_size)
            if current_batch_size <= 0:
                break
            
            class_labels = torch.full((current_batch_size,), class_idx, device=device, dtype=torch.long)
            
            samples = sampler.sample(current_batch_size, class_labels, 
                                   guidance_scale=guidance_scale, use_discriminator=True)
            
            samples = (samples + 1) / 2
            samples = torch.clamp(samples, 0, 1)
            
            samples_np = samples.cpu().numpy()
            samples_np = (samples_np * 255).astype(np.uint8)
            samples_np = np.transpose(samples_np, (0, 2, 3, 1))
            
            all_images.append(samples_np)
            
            labels = np.full(current_batch_size, original_class)
            all_labels.extend(labels)
            
            torch.cuda.empty_cache()
    
    all_images = np.concatenate(all_images, axis=0)
    all_labels = np.array(all_labels)
    
    print(f"Generated: {all_images.shape}")
    
    return all_images, all_labels

def run_diffusion_gan_pipeline():
    
    
    denoiser = load_pretrained_diffusion_model()
    
    discriminator = train_discriminator_for_diffusion(denoiser, epochs=50)
    
    os.makedirs('results/diffusion_gan', exist_ok=True)
    
    torch.save({
        'discriminator': discriminator.state_dict(),
        'denoiser': denoiser.state_dict()
    }, 'results/diffusion_gan/hybrid_model.pth')
    
    synthetic_images, synthetic_labels = generate_hybrid_samples(
        denoiser, discriminator, 
        samples_per_class=200,
        guidance_scale=0.3
    )
    
    np.save('results/diffusion_gan/synthetic_images.npy', synthetic_images)
    np.save('results/diffusion_gan/synthetic_labels.npy', synthetic_labels)
    
    print("\nPIPELINE COMPLETE")
    print(f"Synthetic data: results/diffusion_gan/")
    print(f"Sample images: results/diffusion_gan/samples/")
    
    return denoiser, discriminator

denoiser, discriminator = run_diffusion_gan_pipeline()

Using device: cuda
Loading pre trained diffusion model
Diffusion model loaded

Training discriminator for diffusion GAN hybrid
Training samples: 1486


Preprocessing: 100%|██████████| 800/800 [00:00<00:00, 1177.06it/s]


Epoch 1/50, D Loss: 0.4995
Epoch 2/50, D Loss: 0.3950
Epoch 3/50, D Loss: 0.3556
Epoch 4/50, D Loss: 0.3499
Epoch 5/50, D Loss: 0.3452
Epoch 6/50, D Loss: 0.3401
Epoch 7/50, D Loss: 0.3390
Epoch 8/50, D Loss: 0.3380
Epoch 9/50, D Loss: 0.3378
Epoch 10/50, D Loss: 0.3347
Epoch 11/50, D Loss: 0.3361
Epoch 12/50, D Loss: 0.3316
Epoch 13/50, D Loss: 0.3335
Epoch 14/50, D Loss: 0.3317
Epoch 15/50, D Loss: 0.3323
Epoch 16/50, D Loss: 0.3613
Epoch 17/50, D Loss: 0.3971
Epoch 18/50, D Loss: 0.3495
Epoch 19/50, D Loss: 0.3349
Epoch 20/50, D Loss: 0.3310
Epoch 21/50, D Loss: 0.3304
Epoch 22/50, D Loss: 0.3357
Epoch 23/50, D Loss: 0.3355
Epoch 24/50, D Loss: 0.3293
Epoch 25/50, D Loss: 0.3303
Epoch 26/50, D Loss: 0.3369
Epoch 27/50, D Loss: 0.3340
Epoch 28/50, D Loss: 0.3291
Epoch 29/50, D Loss: 0.3286
Epoch 30/50, D Loss: 0.3275
Epoch 31/50, D Loss: 0.3268
Epoch 32/50, D Loss: 0.3276
Epoch 33/50, D Loss: 0.3267
Epoch 34/50, D Loss: 0.3265
Epoch 35/50, D Loss: 0.3280
Epoch 36/50, D Loss: 0.3281
E

Class 4


Class 8


Class 9


Samples saved to results/diffusion_gan/samples/
Generating class 1


Generating class 4


Generating class 8


Generating class 9


Generated: (800, 128, 128, 3)

PIPELINE COMPLETE
Synthetic data: results/diffusion_gan/
Sample images: results/diffusion_gan/samples/
